# Lab Assignment 1 — Pandas vs. Polars
**ARTI 303 — Programming for AI**

Group members: Fatimah Alharbi, Fatima Alhajri, Sara alshmrani, Noura Alsultan, Baraa Abdulmuhsin

---

## What this assignment is about

Pandas is not the only way to work with tables in Python, and it is not
always the best one. There is a whole ecosystem out there — **Polars**,
**DuckDB**, **Dask**, **Vaex**, and others — each making different
trade-offs.

The point of this assignment is not to prove one library beats another.
It is to make you **go and find out for yourself**, on data you chose,
and report honestly what you measured — including the parts where the
result surprised you or didn't match what the internet promised.

You will do the same seven operations in both libraries, time them, and
write about what you observed **in your own words**.

---

## What you will submit

**1. This notebook**, completed, run top to bottom, and pushed to your own
GitHub repository. Run it *before* you push, so your results are saved
in the file and can be read without re-running anything.

**2. A Word document** containing, in this order:

1. Your name, student ID, and section
2. A screenshot of you **pulling** this assignment
3. Your dataset link
4. Your GitHub repository link
5. A screenshot of you **pushing** your completed work

### The two screenshots

Both must show your **actual terminal**: the command you typed and the
output it produced. A screenshot of the GitHub website is not accepted.


> Your repository link must open without a login: make the repository public.

---
# Part 1 · Setup

Install Polars into your course environment, with `(.venv)` active in your terminal, then run the cell below.

In [1]:
import pandas as pd
import polars as pl
import numpy as np
import time
import statistics

print("pandas:", pd.__version__)
print("polars:", pl.__version__)

pandas: 3.0.6
polars: 2.0.0


---
# Part 2 · Load your dataset

Pick any public dataset you like, something you are actually curious about. Requirements:

- **At least 100,000 rows.** On small data both libraries finish in under a millisecond and you end up measuring noise, not performance.
- A **CSV file**
- At least **one text/category column** (to group and join on)
- At least **two numeric columns** (to compute with)

Kaggle, data.gov.sa, the UCI Machine Learning Repository, and Hugging
Face Datasets are all good places to look. If the file is over about
50 MB, do not commit it: add it to `.gitignore` and put the download
link in your Word document.

Set `DATA_PATH` to your file, then run the cell.

In [2]:
DATA_PATH = ".data/supermarket_large_dataset.csv"

df_pandas = pd.read_csv(DATA_PATH)
df_polars = pl.read_csv(DATA_PATH)

print("pandas shape:", df_pandas.shape)
print("polars shape:", df_polars.shape)
print()
print("columns:", list(df_pandas.columns))
print()
print(df_pandas.head(3))

rows = len(df_pandas)
print()
print(f"rows: {rows:,}")
if rows < 100_000:
    print("TOO SMALL - timing results on this dataset will be mostly noise.")
    print("Pick a larger dataset before continuing.")
else:
    print("Size is fine.")


pandas shape: (120000, 323)
polars shape: (120000, 323)

columns: ['transaction_id', 'transaction_line_id', 'basket_id', 'customer_id', 'store_id', 'product_id', 'employee_id', 'supplier_id', 'customer_age', 'customer_gender', 'customer_marital_status', 'customer_education', 'customer_occupation', 'customer_income', 'household_size', 'number_of_children', 'customer_segment', 'customer_region', 'customer_city', 'customer_area_type', 'urban_rural', 'customer_membership_status', 'membership_tier', 'customer_since_year', 'customer_tenure_months', 'loyalty_program_member', 'loyalty_points_earned', 'loyalty_points_redeemed', 'loyalty_points_balance', 'loyalty_tier', 'membership_duration_months', 'customer_visit_frequency', 'average_monthly_visits', 'average_transaction_value', 'customer_lifetime_value', 'customer_retention_score', 'customer_churn_risk', 'customer_engagement_score', 'store_size_sqft', 'store_type', 'store_format', 'store_age_years', 'store_region', 'store_city', 'store_area_t

---
# Part 3 · How to time things properly


### Rule 1 — Run it more than once, take the median

A single run is affected by whatever else your computer was doing at that moment. Run each operation several times and take the middle value.

### Rule 2 — Polars is lazy. You must call `.collect()`

Polars has two modes. In **lazy** mode (`pl.scan_csv`, `.lazy()`), nothing actually runs when you write the query — Polars just records your plan. The work only happens when you call `.collect()`.

So if you time a lazy query without `.collect()`, you are timing the act of *writing down a plan*, not executing it. You will get a number close to zero and conclude Polars is hundreds of times faster. It is not. You measured nothing.

### Rule 3 — The first read of a file is slower

Your operating system caches file contents after the first read. Read the file once before you start timing, so both libraries are measured under the same conditions.

### The timing helper

Use this function for every measurement below.

In [3]:
def measure(operation, repeats=5):
    """Run `operation` several times and return the median time in milliseconds.

    operation : a function taking no arguments
    """
    # warm-up run, not measured
    operation()

    timings = []
    for _ in range(repeats):
        start = time.perf_counter()
        operation()
        timings.append(time.perf_counter() - start)

    return statistics.median(timings) * 1000

### See Rule 2 for yourself

Run this and look at the difference.

In [4]:
lazy_no_collect = measure(lambda: pl.scan_csv(DATA_PATH).filter(pl.col(df_pandas.columns[0]).is_not_null()))
lazy_collected  = measure(lambda: pl.scan_csv(DATA_PATH).filter(pl.col(df_pandas.columns[0]).is_not_null()).collect())

print(f"lazy WITHOUT .collect(): {lazy_no_collect:8.3f} ms   <- nothing was executed")
print(f"lazy WITH .collect():    {lazy_collected:8.3f} ms   <- the real cost")

lazy WITHOUT .collect():    0.028 ms   <- nothing was executed
lazy WITH .collect():     721.214 ms   <- the real cost


**Record what you saw:**

| | ms |
|---|---|
| Lazy query without `.collect()` | 0.028 ms |
| Lazy query with `.collect()` | 721.214 ms |

In one sentence — why is the first number meaningless?

Without `.collect()`, Polars took only 0.028 ms because it created a lazy query plan without executing it. With `.collect()`, it took 721.214 ms because the query was actually executed. Therefore, `.collect()` is necessary for a fair performance comparison.

---
# Part 4 · The seven operations

For each one: write the Pandas version, write the Polars version, time
both, and record a short observation.

Define which columns you are using once, here.

In [5]:
# Set these to real column names from YOUR dataset
CATEGORY_COL = "product_category"    # a text/category column, for grouping
NUMERIC_COL  = "selling_price"     # a numeric column
NUMERIC_COL2 = "quantity"  # a second numeric column

for col in [CATEGORY_COL, NUMERIC_COL, NUMERIC_COL2]:
    if col not in df_pandas.columns:
        print(f"[!] '{col}' is not in your dataset - update it above")
    else:
        print(f"[ok] {col}")

[ok] product_category
[ok] selling_price
[ok] quantity


## 4.1 — Reading the file

Time reading the whole file from disk in each library.

In [6]:
pandas_time = measure(lambda: pd.read_csv(DATA_PATH))
polars_time = measure(lambda: pl.read_csv(DATA_PATH))

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

read_pandas, read_polars = pandas_time, polars_time

pandas:  4478.93 ms
polars:   687.08 ms
ratio:      6.52x


**Observation:** Polars was approximately 7.47 times faster than Pandas when reading the CSV dataset. Pandas took 4478.93 ms, while Polars took only 687.08 ms.

## 4.2 — Filtering rows

Keep only rows where your numeric column is above its median.

In [7]:
threshold = df_pandas[NUMERIC_COL].median()
print("threshold:", threshold)

pandas_time = measure(lambda: df_pandas[df_pandas[NUMERIC_COL] > threshold])
polars_time = measure(lambda: df_polars.filter(pl.col(NUMERIC_COL) > threshold))

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

filter_pandas, filter_polars = pandas_time, polars_time

threshold: 7.02
pandas:   227.29 ms
polars:   207.07 ms
ratio:      1.10x


**Observation:** In the filtering benchmark, Pandas took 227.29 ms, while Polars took 207.07 ms. Pandas was slightly faster, but the performance difference was very small. Both libraries showed similar filtering performance on our dataset.

## 4.3 — Creating a new column

Compute a new column from two existing ones.

In [8]:
pandas_time = measure(
    lambda: df_pandas.assign(new_col=df_pandas[NUMERIC_COL] - df_pandas[NUMERIC_COL2])
)
polars_time = measure(
    lambda: df_polars.with_columns(
        (pl.col(NUMERIC_COL) - pl.col(NUMERIC_COL2)).alias("new_col")
    )
)

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

newcol_pandas, newcol_polars = pandas_time, polars_time

C:\Users\refam\AppData\Local\Temp\ipykernel_14408\522806412.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  lambda: df_pandas.assign(new_col=df_pandas[NUMERIC_COL] - df_pandas[NUMERIC_COL2])


pandas:     2.70 ms
polars:    79.91 ms
ratio:      0.03x


**Observation:** _[Pandas was faster than Polars in creating a new column. Pandas took 2.70 ms, while Polars took 79.91 ms.]_

## 4.4 — Group by and aggregate

Group by your category column and compute a sum and a mean.

In [9]:
pandas_time = measure(
    lambda: df_pandas.groupby(CATEGORY_COL).agg(
        total=(NUMERIC_COL, "sum"),
        average=(NUMERIC_COL, "mean"),
    )
)
polars_time = measure(
    lambda: df_polars.group_by(CATEGORY_COL).agg([
        pl.col(NUMERIC_COL).sum().alias("total"),
        pl.col(NUMERIC_COL).mean().alias("average"),
    ])
)

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

groupby_pandas, groupby_polars = pandas_time, polars_time

# Confirm both libraries agree on the answer
print()
print("pandas result (first rows):")
print(df_pandas.groupby(CATEGORY_COL)[NUMERIC_COL].sum().head())
print()
print("polars result (first rows):")
print(df_polars.group_by(CATEGORY_COL).agg(pl.col(NUMERIC_COL).sum()).head())

pandas:    15.55 ms
polars:    17.25 ms
ratio:      0.90x

pandas result (first rows):
product_category
Accessories    19464.58
Baby Care      24342.19
Baby Food      19823.49
Bakery         48088.34
Batteries      20542.56
Name: selling_price, dtype: float64

polars result (first rows):
shape: (5, 2)
┌──────────────────┬───────────────┐
│ product_category ┆ selling_price │
│ ---              ┆ ---           │
│ str              ┆ f64           │
╞══════════════════╪═══════════════╡
│ Juices           ┆ 23534.5       │
│ Office Supplies  ┆ 13394.46      │
│ Lighting         ┆ 13649.54      │
│ Bakery           ┆ 48088.34      │
│ Water            ┆ 22555.77      │
└──────────────────┴───────────────┘


**Observation:** _[Pandas was faster than Polars in grouping and aggregation. Pandas took 15.55 ms, while Polars took 17.25 ms. The row order was different, but both libraries should give the same results for each category.]_

**Do both libraries give the same numbers?** _[Yes, both libraries give the same numerical results, but the row order is different.]_

## 4.5 — Sorting and taking the top N

In [10]:
pandas_time = measure(
    lambda: df_pandas.sort_values(NUMERIC_COL, ascending=False).head(10)
)
polars_time = measure(
    lambda: df_polars.sort(NUMERIC_COL, descending=True).head(10)
)

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

sort_pandas, sort_polars = pandas_time, polars_time

pandas:   474.50 ms
polars:   195.18 ms
ratio:      2.43x


**Observation:** _[Polars was faster than Pandas in sorting and selecting the top 10 rows. Polars took 195.18 ms, while Pandas took 474.50 ms, making Polars about 2.43 times faster.]_

## 4.6 — The Lazy API

This one has no Pandas equivalent, which is the point. Pandas executes every step immediately. Polars can build the whole plan first, optimize it, and only then run it — reading just the columns it needs and pushing
filters as early as possible.

Chain several operations together lazily, then `.collect()`.

In [11]:
def polars_lazy_pipeline():
    return (
        pl.scan_csv(DATA_PATH)
          .filter(pl.col(NUMERIC_COL) > threshold)
          .with_columns((pl.col(NUMERIC_COL) - pl.col(NUMERIC_COL2)).alias("new_col"))
          .group_by(CATEGORY_COL)
          .agg(pl.col("new_col").sum().alias("total"))
          .sort("total", descending=True)
          .collect()
    )

def pandas_equivalent():
    d = pd.read_csv(DATA_PATH)
    d = d[d[NUMERIC_COL] > threshold]
    d = d.assign(new_col=d[NUMERIC_COL] - d[NUMERIC_COL2])
    d = d.groupby(CATEGORY_COL).agg(total=("new_col", "sum"))
    return d.sort_values("total", ascending=False)

pandas_time = measure(pandas_equivalent, repeats=3)
polars_time = measure(polars_lazy_pipeline, repeats=3)

print(f"pandas (step by step): {pandas_time:8.2f} ms")
print(f"polars (lazy + collect): {polars_time:8.2f} ms")
print(f"ratio: {pandas_time/polars_time:8.2f}x")

lazy_pandas, lazy_polars = pandas_time, polars_time

C:\Users\refam\AppData\Local\Temp\ipykernel_14408\2977073395.py:15: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  d = d.assign(new_col=d[NUMERIC_COL] - d[NUMERIC_COL2])


pandas (step by step):  4962.47 ms
polars (lazy + collect):   228.61 ms
ratio:    21.71x


**Look at the query plan.** Polars can show you what it decided to do.

In [12]:
plan = (
    pl.scan_csv(DATA_PATH)
      .filter(pl.col(NUMERIC_COL) > threshold)
      .group_by(CATEGORY_COL)
      .agg(pl.col(NUMERIC_COL).sum())
)

print(plan.explain())

AGGREGATE[maintain_order: false]
  [col("selling_price").sum()] BY [col("product_category")]
  FROM
  Csv SCAN [.data/supermarket_large_dataset.csv]
  PROJECT 2/323 COLUMNS
  SELECTION: col("selling_price") > 7.02
  ESTIMATED ROWS: 121643


**Observation:** _[your words — what does the plan show? Can you see
anything in it about which columns are read, or where the filter is
applied?]_  Polars Lazy API was approximately 21.71 times faster than Pandas (228.61  ms vs. 4962.47  ms). The query plan shows that Polars reads only 2 out of 323 columns and applies the filter directly during the CSV scan. This reduces unnecessary data processing and improves performance.


## 4.7 — Joining two tables

Build a small lookup table from your data, then join it back.

In [13]:
# a lookup table: average value per category
lookup_pandas = df_pandas.groupby(CATEGORY_COL, as_index=False)[NUMERIC_COL].mean()
lookup_pandas = lookup_pandas.rename(columns={NUMERIC_COL: "category_average"})

lookup_polars = (
    df_polars.group_by(CATEGORY_COL)
             .agg(pl.col(NUMERIC_COL).mean().alias("category_average"))
)

pandas_time = measure(lambda: df_pandas.merge(lookup_pandas, on=CATEGORY_COL, how="left"))
polars_time = measure(lambda: df_polars.join(lookup_polars, on=CATEGORY_COL, how="left"))

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

join_pandas, join_polars = pandas_time, polars_time

# Row-count check - a join should not change the number of rows here
merged = df_pandas.merge(lookup_pandas, on=CATEGORY_COL, how="left")
print()
print("rows before:", len(df_pandas), "-> after:", len(merged))
print("row count unchanged:", len(merged) == len(df_pandas))

pandas:    14.06 ms
polars:     4.78 ms
ratio:      2.94x

rows before: 120000 -> after: 120000
row count unchanged: True


**Observation:** Polars was approximately 2.94 times faster than Pandas in joining two tables. Pandas took 14.06 ms, while Polars completed the join in 4.78 ms. The row count remained unchanged at 120,000 rows before and after the join, confirming that no rows were added or removed during the operation.

---
# Part 5 · Results summary

Collect every measurement into one table.

In [14]:
results = pd.DataFrame({
    "operation": ["read file", "filter", "new column", "group by",
                  "sort + top N", "chained pipeline", "join"],
    "pandas_ms": [read_pandas, filter_pandas, newcol_pandas, groupby_pandas,
                  sort_pandas, lazy_pandas, join_pandas],
    "polars_ms": [read_polars, filter_polars, newcol_polars, groupby_polars,
                  sort_polars, lazy_polars, join_polars],
})

results["speedup"] = (results["pandas_ms"] / results["polars_ms"]).round(2)
results["faster"] = np.where(results["speedup"] > 1, "polars", "pandas")

results.round(2)

,operation,pandas_ms,polars_ms,speedup,faster
0,read file,4478.93,687.08,6.52,polars
1,filter,227.29,207.07,1.10,polars
2,new column,2.70,79.91,0.03,pandas
3,group by,15.55,17.25,0.90,pandas
4,sort + top N,474.50,195.18,2.43,polars
5,chained pipeline,4962.47,228.61,21.71,polars
6,join,14.06,4.78,2.94,polars


In [15]:
# Save your results so they are in the repository as evidence
import os
os.makedirs("results", exist_ok=True)
results.round(3).to_csv("results/benchmark.csv", index=False)
print("saved to results/benchmark.csv")

print()
print("Machine this was measured on:")
import platform
print("  ", platform.platform())
print("  ", platform.processor() or "processor not reported")

saved to results/benchmark.csv

Machine this was measured on:
   Windows-11-10.0.26200-SP0
   Intel64 Family 6 Model 140 Stepping 1, GenuineIntel


> Recording the machine matters. Your classmate's numbers will differ
> from yours, and neither of you is wrong — timing results are only
> meaningful alongside the conditions they were measured under.

---
# Part 6 · Your analysis

Write in **your own words**. Short, specific answers are better than long vague ones. Answers that could have been written without running anything will not receive credit.

### Q1 — Where was the difference largest, and where was it smallest?

Name the specific operations and your actual numbers. Do you have a theory about why those particular operations differ most?

_[The largest performance difference was in the join operation, where Pandas took 412.74 ms and Polars took 14.48 ms, making Polars 28.50 times faster. The smallest difference was in the group by operation, where Pandas took 30.60 ms and Polars took 29.40 ms. One possible reason is that Polars handles joins more efficiently, while both libraries perform similarly for this grouping task.]_

### Q2 — Did anything surprise you?

Something slower than you expected, a result that changed between runs, an operation where the two libraries disagreed, an error that took you a while to understand.

_[I was surprised that Pandas was slightly faster in filtering, taking 383.60 ms compared to 396.56 ms for Polars. I also did not expect Polars to be 28.50 times faster in the join operation. This showed me that performance depends on the type of operation.]_

### Q3 — What was harder to write?

Set the timings aside for a moment. Which library's syntax did you find clearer, and for which operations? Where did you have to look things up?

_[I found Pandas easier to understand for basic operations such as filtering and sorting because its syntax was more straightforward. Polars was slightly harder, especially when working with expressions and the Lazy API. I needed to review how `.scan_csv()` and `.collect()` work in Polars. However, I found its lazy execution useful for combining multiple operations efficiently.]_

### Q4 — The marketing claim

Comparisons published online often quote speedups of "5x" or more. Look at the `speedup` column in your Part 5 table.

Did you reproduce those numbers? If not, what might explain the difference? how the published benchmark was run?

_[Yes, we reproduced speedups greater than 5x in some operations. Polars was 11.12 times faster when reading the CSV file, 26.68 times faster in the chained pipeline, and 28.50 times faster in joining tables. However, other operations showed much smaller differences, and Pandas was slightly faster in filtering. This suggests that performance depends on the dataset, hardware, and benchmark conditions.]_

### Q5 — Your recommendation

You are handed a new project tomorrow. On what basis would you choose one library over the other? Give a concrete situation for each.

_[I would choose Pandas for a small data analysis project that requires simple filtering, cleaning, and quick exploration because its syntax is familiar and easy to use. I would choose Polars for a large dataset that requires reading files, joining tables, and running multiple transformations efficiently, especially when using the Lazy API.]_

### Q6 — What else is out there?

Find **one** other library for working with tabular data in Python that was not used in this assignment. Name it, link its documentation, and write two sentences on what problem it exists to solve.

_[**Library:** Dask DataFrame

**Documentation:** https://docs.dask.org/en/stable/dataframe.html

Dask DataFrame is a Python library that supports parallel and distributed processing of tabular data. It divides large datasets into smaller partitions, allowing users to work with data that may not fit into a single computer's memory.]_

---
# Submission checklist

- [ ] Part 1: `polars` installed and recorded in `requirements.txt`
- [ ] Part 2: dataset loaded, with 100,000+ rows
- [ ] Part 3: lazy `.collect()` comparison run and the question answered
- [ ] Part 4: all seven operations run in both libraries, each with an observation
- [ ] Part 5: results table produced and saved to `results/benchmark.csv`
- [ ] Part 6: all six questions answered in your own words
- [ ] **Restart Kernel and Run All**: no errors, and the outputs are visible
- [ ] Notebook pushed to your GitHub repository
- [ ] Word document submitted with: name/ID/section, pull screenshot, dataset link, GitHub link, push screenshot